# CSE 151B Post-Training Curriculum Pipeline

Fresh Lightning AI notebook for restarting Qwen3-4B-Thinking-2507 post-training with the updated curriculum.

All artifacts are saved under the Lightning workspace.


## Run Order

1. Setup, paths, prompts, helpers.
2. Build evaluation set once.
3. Check Stage 0B format-anchor data.
4. Build and inspect the Stage 1-2 dataset.
5. Stage 1-2 trial: 400 steps, evaluate gate.
6. Stage 1-2 full: train, evaluate, merge.
7. Stage 2.5 rejection sampling: generate correct self-traces, train, evaluate, merge.
8. Stage 3-4 hard curriculum: train, evaluate, merge.
9. Stage 3.5 hard rejection sampling: train, evaluate, merge.
10. Stage 5 replay/stabilization: train, evaluate, final merge.


In [1]:
# Optional one-time dependency install for Lightning AI.
# Run only if your current environment does not already have these packages.
# Training/SFT uses Unsloth; eval and rejection sampling use vLLM offline inference.
# After installing, restart the kernel before importing torch/unsloth/vllm.
# %pip install -U unsloth trl peft accelerate bitsandbytes datasets transformers sentencepiece protobuf scipy tqdm sympy
# vLLM 0.19.0 pins torch==2.10.0; do not add vllm-flash-attn==2.6.1 because it pins torch==2.4.0.
# vLLM's OpenCV dependency installs NumPy 2.x on Python 3.12, so keep SciPy/sklearn/pandas new enough for NumPy 2.
# %pip install -U "vllm==0.19.0" "scipy>=1.15,<1.17" "scikit-learn>=1.5,<1.8" "pandas>=2.2,<2.4"


In [2]:
# import os

# os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')
# os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# import vllm
# from vllm import LLM, SamplingParams

# print(f"vLLM is ready: {vllm.__version__}")


In [3]:
import gc
import json
import math
import os
import random
import re
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import torch
from datasets import Dataset, concatenate_datasets, load_dataset, load_from_disk
from tqdm.auto import tqdm

ROOT = Path('/teamspace/studios/this_studio') if Path('/teamspace/studios/this_studio').exists() else Path.cwd()
COMP_DIR = ROOT / 'comp_folder/151B_SP26_Competition'
PUBLIC_PATH = COMP_DIR / 'data/public.jsonl'
STARTER_RESULTS_PATH = COMP_DIR / 'results/12k_thinking/starter_results_12k.jsonl'

ARTIFACT_DIR = ROOT / 'artifacts/post_training_curriculum'
DATASET_DIR = ARTIFACT_DIR / 'datasets'
EVAL_DIR = ARTIFACT_DIR / 'eval'
CHECKPOINT_DIR = ROOT / 'checkpoints'
RESULTS_LOG_PATH = CHECKPOINT_DIR / 'results_log.json'

for path in [ARTIFACT_DIR, DATASET_DIR, EVAL_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(COMP_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(COMP_DIR.resolve()))

try:
    from judger import Judger
    judger = Judger(strict_extract=False)
except Exception as exc:
    print(f'Warning: competition Judger unavailable: {exc}')
    judger = None

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL = 'Qwen/Qwen3-4B-Thinking-2507'
MAX_SEQ_LENGTH = 8192
MAX_NEW_TOKENS = 2048
EVAL_MAX_NEW_TOKENS = 1024
EVAL_VLLM_TEMPERATURE = 0.0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('ROOT:', ROOT)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


ROOT: /home/lucasyan/qwen3-4b-finetuning
CUDA available: True
GPU: NVIDIA L4


In [4]:
SYSTEM_PROMPT = """You are an expert mathematician. Solve the problem step by step and put your final answer within \\boxed{}.

For multi-part questions with [ANS] blanks: put all answers comma-separated in one \\boxed{}.
For MCQ: identify the correct option and put only the letter in \\boxed{}.
Never round intermediate calculations.
"""

EVAL_SYSTEM_PROMPT = SYSTEM_PROMPT

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# vLLM is used only for inference-heavy eval/rejection sampling.
# Keep these conservative for a single 24GB-ish GPU; raise utilization/batch size if stable.
VLLM_GPU_MEMORY_UTILIZATION = 0.82
VLLM_TENSOR_PARALLEL_SIZE = 1
VLLM_DTYPE = "auto"
VLLM_BATCH_SIZE = 20
VLLM_ENFORCE_EAGER = True

# Sampling params matching Qwen3-Thinking-2507 generation_config.json.
# temperature=0 with no repetition_penalty causes degenerate repetition loops on this model.
VLLM_TEMPERATURE = 0.6
VLLM_TOP_P = 0.95
VLLM_TOP_K = 20
VLLM_REPETITION_PENALTY = 1.25

@dataclass
class StageConfig:
    name: str
    lr: float
    epochs: Optional[float] = 1.0
    max_steps: int = -1
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.0
    batch_size: int = 1
    grad_accum: int = 16
    warmup_ratio: float = 0.03
    scheduler: str = "cosine"

STAGE0 = StageConfig(name="stage0", lr=3e-5, epochs=None, max_steps=200, lora_r=64, lora_alpha=128)
STAGE1_2_TRIAL = StageConfig(name="stage1_2_trial", lr=1e-5, epochs=None, max_steps=400, lora_r=64, lora_alpha=128)
STAGE1_2 = StageConfig(name="stage1_2", lr=1e-5, epochs=1.0, max_steps=-1, lora_r=64, lora_alpha=128)
STAGE2_5 = StageConfig(name="stage2_5", lr=1e-5, epochs=1.0, max_steps=-1, lora_r=64, lora_alpha=128)
STAGE3_4 = StageConfig(name="stage3_4", lr=8e-6, epochs=2.0, max_steps=-1, lora_r=64, lora_alpha=128)
STAGE3_5 = StageConfig(name="stage3_5", lr=8e-6, epochs=1.0, max_steps=-1, lora_r=64, lora_alpha=128)
STAGE5 = StageConfig(name="stage5", lr=3e-6, epochs=1.0, max_steps=-1, lora_r=32, lora_alpha=64)


In [5]:
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows: Sequence[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def load_cached_jsonl(path: Path, force_rebuild: bool = False) -> Optional[List[Dict[str, Any]]]:
    if force_rebuild or not path.exists():
        return None
    rows = load_jsonl(path)
    print(f'Loaded cached dataset: {len(rows)} records from {path}')
    return rows


def parse_mathqa_options(raw_options: Any) -> List[str]:
    text = str(raw_options or '').strip()
    if not text:
        return []
    parts = re.split(r'\s*[a-e]\s*\)', text, flags=re.I)
    return [part.strip().strip(',') for part in parts if part.strip().strip(',')]


def load_mathqa_train():
    # The main allenai/math_qa repo can fail on newer datasets versions because it still has a legacy loader script.
    # This revision contains parquet files, so it loads without executing the old math_qa.py script.
    try:
        return load_dataset(
            'allenai/math_qa',
            'default',
            split='train',
            revision='795ca7da22406a2d62cc8874d0f2b427386b68db',
        )
    except Exception as exc:
        print(f'Warning: could not load MathQA parquet revision from HF: {exc}')

    import urllib.request
    import zipfile

    zip_path = DATASET_DIR / 'MathQA.zip'
    if not zip_path.exists():
        print(f'Downloading official MathQA zip to {zip_path}')
        urllib.request.urlretrieve('https://math-qa.github.io/math-QA/data/MathQA.zip', zip_path)

    with zipfile.ZipFile(zip_path) as zf:
        train_members = [name for name in zf.namelist() if name.endswith('train.json')]
        if not train_members:
            raise FileNotFoundError(f'No train.json found inside {zip_path}')
        with zf.open(train_members[0]) as f:
            return json.load(f)


def option_labels(n: int) -> List[str]:
    return [chr(65 + i) for i in range(n)]


def format_options(options: Optional[Sequence[str]]) -> str:
    if not options:
        return ''
    return '\n'.join(f'{label}. {str(opt).strip()}' for label, opt in zip(option_labels(len(options)), options))


def build_user_problem(question: str, options: Optional[Sequence[str]] = None) -> str:
    if options:
        return f'{question}\n\nOptions:\n{format_options(options)}'
    return question


def count_ans_blanks(question: str) -> int:
    return str(question).count('[ANS]')


def normalize_final_answer(answer: Any) -> str:
    if isinstance(answer, (list, tuple)):
        return ', '.join(str(x).strip() for x in answer)
    return str(answer).strip()


def strip_outer_boxed(text: str) -> str:
    boxed = extract_last_boxed(text)
    return boxed if boxed else str(text).strip()


def make_assistant_content(reasoning: str, final_answer: Any) -> str:
    final_raw = normalize_final_answer(final_answer)
    final_boxes = [box.strip() for box in extract_all_boxed(final_raw) if box.strip()]
    final = final_boxes[-1] if final_boxes else re.sub(r'\\boxed\s*\{\s*\}', '', final_raw).strip()
    reasoning = re.sub(r'</?think>\s*', '', str(reasoning or '')).strip()
    if not reasoning:
        reasoning = 'I solve the problem carefully and keep the final answer in the required format.'
    return f'<think>\n{reasoning}\n</think>\n\n\\boxed{{{final}}}'


def make_chat_text(tokenizer, question: str, answer: Any, reasoning: str, options: Optional[Sequence[str]] = None) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_problem(question, options)},
        {'role': 'assistant', 'content': make_assistant_content(reasoning, answer)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def extract_all_boxed(text: str) -> List[str]:
    text = text or ''
    boxes = []
    for match in re.finditer(r'\\boxed\s*\{', text):
        start = match.end()
        depth = 1
        i = start
        while i < len(text) and depth:
            if text[i] == '{':
                depth += 1
            elif text[i] == '}':
                depth -= 1
            i += 1
        if depth == 0:
            boxes.append(text[start:i - 1].strip())
    return boxes


def extract_last_boxed(text: str) -> str:
    boxes = extract_all_boxed(text)
    return boxes[-1] if boxes else ''


def extract_after_think(text: str) -> str:
    parts = (text or '').split('</think>')
    return parts[-1].strip() if len(parts) > 1 else (text or '').strip()


def extract_mcq_letter(text: str, valid_n: int = 10) -> str:
    valid = set(option_labels(valid_n))
    boxed = extract_last_boxed(text).strip().upper()
    if boxed in valid:
        return boxed
    matches = re.findall(r'\b([A-Z])\b', (text or '').upper())
    matches = [m for m in matches if m in valid]
    return matches[-1] if matches else ''


def split_final_answers(answer_text: str) -> List[str]:
    text = strip_outer_boxed(answer_text)
    if not text:
        return []
    # Public gold values do not use comma-containing nested structures except intervals/CI.
    # If the whole answer is a parenthesized interval, keep it as one answer.
    if text.startswith('(') and text.endswith(')'):
        return [text.replace(' ', '')]
    return [part.strip() for part in text.split(',') if part.strip()]


def boxed_format_ok(question: str, response: str, is_mcq: bool = False, options: Optional[Sequence[str]] = None) -> bool:
    after = extract_after_think(response)
    boxes_after = extract_all_boxed(after)
    if len(boxes_after) != 1:
        return False
    final = boxes_after[0].strip()
    if is_mcq:
        return final.upper() in set(option_labels(len(options or [])))
    expected = count_ans_blanks(question)
    if expected <= 1:
        return bool(final)
    return len(split_final_answers(final)) == expected



In [6]:
def score_public_item(item: Dict[str, Any], response: str) -> bool:
    gold = item.get('answer')
    options = item.get('options')
    if options:
        return extract_mcq_letter(response, valid_n=len(options)) == str(gold).strip().upper()
    gold_list = gold if isinstance(gold, list) else [gold]
    if judger is not None:
        try:
            return bool(judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list)))
        except Exception as exc:
            print(f'Judger failed for id={item.get("id")}: {exc}')
    pred_parts = split_final_answers(response)
    return [p.replace(' ', '') for p in pred_parts] == [str(g).replace(' ', '') for g in gold_list]


def score_math_item(item: Dict[str, Any], response: str) -> bool:
    gold = normalize_final_answer(item.get('answer', ''))
    pred = strip_outer_boxed(response)
    if judger is not None:
        try:
            return bool(judger.auto_judge(pred=response, gold=[gold], options=[[]]))
        except Exception:
            pass
    return pred.replace(' ', '') == gold.replace(' ', '')


def response_word_count(text: str) -> int:
    return len(re.findall(r'\S+', text or ''))


def append_results_log(stage_name: str, metrics: Dict[str, Any]) -> None:
    if RESULTS_LOG_PATH.exists():
        log = json.loads(RESULTS_LOG_PATH.read_text())
    else:
        log = []
    row = {'stage': stage_name, **metrics}
    log.append(row)
    RESULTS_LOG_PATH.write_text(json.dumps(log, indent=2, ensure_ascii=False))
    print(f'Updated results log: {RESULTS_LOG_PATH}')


In [7]:
def extract_answer_from_solution(solution: str) -> str:
    boxed = extract_last_boxed(solution)
    if boxed:
        return boxed
    # MATH often uses "The answer is ..." near the end; fallback to the final nonempty line.
    lines = [ln.strip() for ln in str(solution).splitlines() if ln.strip()]
    return lines[-1] if lines else ''


def math_level_value(example: Dict[str, Any]) -> Optional[int]:
    raw = str(example.get('level', example.get('difficulty', ''))).lower()
    m = re.search(r'(\d+)', raw)
    return int(m.group(1)) if m else None


def make_training_record(question: str, answer: Any, reasoning: str, options: Optional[Sequence[str]] = None, source: str = '') -> Dict[str, Any]:
    return {
        'question': question,
        'options': list(options) if options else None,
        'answer': normalize_final_answer(answer),
        'reasoning': reasoning or '',
        'source': source,
        'n_ans': count_ans_blanks(question),
        'is_mcq': bool(options),
    }


def clean_stage0_assistant_content(text: str) -> str:
    text = str(text or '')
    replacements = [
        (r'\s*This confirms the requested (?:value|answer|expression|computation|result)\.\s*', ' '),
        (r'\s*The computation also checks the ordering requested, which is enough to identify the requested result\.\s*', ' The final answer is listed in the required order. '),
        (r'\s*The computation also checks the ordering requested\.\s*', ' The final answer is listed in order. '),
        (r'\brequested\b', 'required'),
    ]
    for pattern, repl in replacements:
        text = re.sub(pattern, repl, text, flags=re.I)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def make_assistant_only_features(tokenizer, row: Dict[str, Any]) -> Dict[str, Any]:
    prompt_text = tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_problem(row['question'], row.get('options'))},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    # Prefer the pre-built target field (teacher output, no injected Self-check) when available.
    # Fall back to make_chat_text only for records that lack a target (e.g. MATH / NuminaMath).
    if row.get('target'):
        assistant_content = clean_stage0_assistant_content(row['target'])
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_problem(row['question'], row.get('options'))},
            {'role': 'assistant', 'content': assistant_content},
        ]
        full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        assistant_content = clean_stage0_assistant_content(make_assistant_content(row.get('reasoning', ''), row.get('target_answer', row.get('answer', ''))))
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_problem(row['question'], row.get('options'))},
            {'role': 'assistant', 'content': assistant_content},
        ]
        full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    if not full_text.startswith(prompt_text):
        raise ValueError('Assistant-only prompt is not a prefix of the full training text')

    full_ids = tokenizer(full_text, add_special_tokens=False).input_ids
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        **row,
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
        'length': len(full_ids),
    }


def format_records_with_tokenizer(records: Sequence[Dict[str, Any]], tokenizer) -> Dataset:
    formatted = []
    skipped = []
    for idx, row in enumerate(records, 1):
        item = make_assistant_only_features(tokenizer, row)
        if len(item['input_ids']) > MAX_SEQ_LENGTH:
            skipped.append((idx, len(item['input_ids']), row.get('distill_key')))
            continue
        formatted.append(item)
    if skipped:
        print(f'Skipped {len(skipped)} over-length records; first skipped: {skipped[0]}')
    print(f'Assistant-only train records: {len(formatted)} / {len(records)}')
    return Dataset.from_list(formatted)


def make_causal_lm_collator(tokenizer):
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    def collate(features: Sequence[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f['input_ids']) for f in features)
        input_ids, attention_mask, labels = [], [], []
        for f in features:
            pad_len = max_len - len(f['input_ids'])
            input_ids.append(f['input_ids'] + [pad_id] * pad_len)
            attention_mask.append(f['attention_mask'] + [0] * pad_len)
            labels.append(f['labels'] + [-100] * pad_len)
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
        }

    return collate


def safe_shuffle_take(rows: Sequence[Dict[str, Any]], n: int, seed: int = SEED) -> List[Dict[str, Any]]:
    rows = list(rows)
    rng = random.Random(seed)
    rng.shuffle(rows)
    return rows[:min(n, len(rows))]


In [8]:
def normalize_tokenizer_special_tokens(tokenizer):
    eos_candidates = [tokenizer.eos_token, '<|im_end|>', '<|endoftext|>']
    eos_token = None
    for token in eos_candidates:
        if token is None:
            continue
        token_id = tokenizer.convert_tokens_to_ids(token)
        if token_id is not None:
            eos_token = token
            break
    if eos_token is None:
        raise ValueError('Could not find a valid EOS token in tokenizer vocabulary')
    tokenizer.eos_token = eos_token
    tokenizer.pad_token = eos_token
    tokenizer.padding_side = 'right'
    return tokenizer


def load_tokenizer_only(model_name_or_path: str = BASE_MODEL):
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)
    return normalize_tokenizer_special_tokens(tokenizer)


def load_model_for_training(model_name_or_path: str, cfg: StageConfig):
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(model_name_or_path),
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        trust_remote_code=True,
    )
    tokenizer = normalize_tokenizer_special_tokens(tokenizer)
    model = FastLanguageModel.get_peft_model(
        model,
        r=cfg.lora_r,
        target_modules=TARGET_MODULES,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias='none',
        use_gradient_checkpointing='unsloth',  # Unsloth handles GC; don't also set in TrainingArguments
        random_state=3407,
    )
    return model, tokenizer


def train_stage(model_name_or_path: str, raw_records: Sequence[Dict[str, Any]], cfg: StageConfig, merge: bool = True) -> Tuple[Path, Optional[Path]]:
    from transformers import Trainer, TrainingArguments

    print(f'Loading model for {cfg.name}: {model_name_or_path}')
    model, tokenizer = load_model_for_training(model_name_or_path, cfg)
    print(f'Tokenizer EOS/PAD: {tokenizer.eos_token!r} / {tokenizer.pad_token!r}')
    train_dataset = format_records_with_tokenizer(raw_records, tokenizer)

    lora_dir = CHECKPOINT_DIR / f'lora_{cfg.name}'
    merged_dir = CHECKPOINT_DIR / f'merged_{cfg.name}'
    output_dir = CHECKPOINT_DIR / f'trainer_{cfg.name}'

    args = TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=cfg.lr,
        num_train_epochs=cfg.epochs if cfg.max_steps == -1 else 1,
        max_steps=cfg.max_steps,
        warmup_ratio=cfg.warmup_ratio,
        lr_scheduler_type=cfg.scheduler,
        optim='adamw_8bit',
        bf16=True,
        fp16=False,
        logging_steps=5,
        save_strategy='no',
        report_to='none',
        seed=SEED,
        gradient_checkpointing=False,  # Unsloth sets use_gradient_checkpointing='unsloth' in get_peft_model; don't double-enable
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        data_collator=make_causal_lm_collator(tokenizer),
        args=args,
    )

    trainer.train()
    lora_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(lora_dir))
    tokenizer.save_pretrained(str(lora_dir))
    print(f'Saved LoRA adapter: {lora_dir}')

    merged_path = None
    if merge:
        merged_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained_merged(str(merged_dir), tokenizer, save_method='merged_16bit')
        merged_path = merged_dir
        print(f'Saved merged 16-bit model: {merged_dir}')

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return lora_dir, merged_path


In [9]:
def build_eval_sets(seed: int = SEED, public_n: int = 50, each_math_level_n: int = 50) -> List[Dict[str, Any]]:
    eval_rows = []
    rng = random.Random(seed)

    if PUBLIC_PATH.exists():
        public = load_jsonl(PUBLIC_PATH)
        public_sample = rng.sample(public, min(public_n, len(public)))
        for row in public_sample:
            eval_rows.append({'eval_source': 'public', **row})
    else:
        print(f'Warning: missing public eval path {PUBLIC_PATH}')

    try:
        math_test = load_dataset('HuggingFaceH4/MATH', split='test')
        easy, medium, hard = [], [], []
        for math_idx, ex in enumerate(math_test):
            lvl = math_level_value(ex)
            answer = extract_answer_from_solution(ex.get('solution', ''))
            row = {
                'id': f"math_test_{ex.get('type','unknown')}_{math_idx}",
                'question': ex.get('problem', ''),
                'answer': answer,
                'options': None,
                'eval_source': 'math_test',
                'level': lvl,
            }
            if lvl is not None and lvl <= 2:
                easy.append(row)
            elif lvl == 3:
                medium.append(row)
            elif lvl is not None and lvl >= 4:
                hard.append(row)
        eval_rows.extend(safe_shuffle_take(easy, each_math_level_n, seed + 1))
        eval_rows.extend(safe_shuffle_take(medium, each_math_level_n, seed + 2))
        eval_rows.extend(safe_shuffle_take(hard, each_math_level_n, seed + 3))
    except Exception as exc:
        print(f'Warning: could not load MATH test eval set: {exc}')

    write_jsonl(eval_rows, EVAL_DIR / 'heldout_eval_set.jsonl')
    print(f'Built eval set with {len(eval_rows)} rows')
    return eval_rows


def generate_response(model, tokenizer, item: Dict[str, Any], max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """Legacy single-item Unsloth generation helper for ad hoc smoke tests."""
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
    prompt = tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_problem(item['question'], item.get('options'))},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def require_vllm() -> None:
    try:
        import vllm  # noqa: F401
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            'vLLM is required for evaluate_model() and run_rejection_sampling(). '
            'Install it in a fresh kernel, for example: %pip install -U "vllm==0.19.0" "scipy>=1.15,<1.17" "scikit-learn>=1.5,<1.8" "pandas>=2.2,<2.4", then restart the kernel.'
        ) from exc


def build_vllm_prompt(tokenizer, item: Dict[str, Any], system_prompt: str = EVAL_SYSTEM_PROMPT) -> str:
    return tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': build_user_problem(item['question'], item.get('options'))},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


def fix_tokenizer_regex(model_path: str) -> None:
    """Restore BASE_MODEL tokenizer files into a local merged checkpoint before vLLM eval."""
    import json as _json
    import shutil as _shutil
    from pathlib import Path as _Path
    from transformers import AutoTokenizer
    from transformers.utils import cached_file

    model_path = _Path(model_path)
    if not model_path.exists():
        return

    base_tokenizer_json = cached_file(BASE_MODEL, 'tokenizer.json', trust_remote_code=True)
    base_tokenizer_config = cached_file(BASE_MODEL, 'tokenizer_config.json', trust_remote_code=True)

    # Copy the full authoritative tokenizer graph. Copying only one field is fragile because
    # AutoTokenizer/save_pretrained can regenerate the pre-tokenizer regex.
    _shutil.copy2(base_tokenizer_json, model_path / 'tokenizer.json')

    with open(base_tokenizer_config) as f:
        tokenizer_config = _json.load(f)
    tokenizer_config['eos_token'] = '<|im_end|>'
    tokenizer_config['pad_token'] = '<|im_end|>'
    with open(model_path / 'tokenizer_config.json', 'w') as f:
        _json.dump(tokenizer_config, f, ensure_ascii=False, indent=2)

    # Keep special-token metadata consistent and avoid Unsloth's extra <|PAD_TOKEN|>.
    special_tokens_map = {
        'bos_token': '<|endoftext|>',
        'eos_token': '<|im_end|>',
        'pad_token': '<|im_end|>',
        'additional_special_tokens': tokenizer_config.get('additional_special_tokens', []),
    }
    with open(model_path / 'special_tokens_map.json', 'w') as f:
        _json.dump(special_tokens_map, f, ensure_ascii=False, indent=2)

    tok = AutoTokenizer.from_pretrained(str(model_path), trust_remote_code=True)
    tok = normalize_tokenizer_special_tokens(tok)

    config_path = model_path / 'config.json'
    if config_path.exists():
        with open(config_path) as f:
            config = _json.load(f)
        config['eos_token_id'] = tok.eos_token_id
        config['pad_token_id'] = tok.pad_token_id
        with open(config_path, 'w') as f:
            _json.dump(config, f, indent=2)

    print(f'Restored BASE_MODEL tokenizer into {model_path}; EOS/PAD ids: {tok.eos_token_id}/{tok.pad_token_id}')
def load_vllm_engine(model_name_or_path: str):
    # vLLM can be initialized after CUDA has been touched by training, but spawn is safer in notebooks.
    os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')
    os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    require_vllm()

    from transformers import AutoTokenizer
    from vllm import LLM

    model_path = str(model_name_or_path)

    # Local Unsloth-merged checkpoints may save a broken pre-tokenizer; fix before vLLM loads it.
    if Path(model_path).exists():
        fix_tokenizer_regex(model_path)

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tokenizer = normalize_tokenizer_special_tokens(tokenizer)

    llm = LLM(
        model=model_path,
        tokenizer=model_path,
        trust_remote_code=True,
        dtype=VLLM_DTYPE,
        max_model_len=MAX_SEQ_LENGTH,
        tensor_parallel_size=VLLM_TENSOR_PARALLEL_SIZE,
        gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
        enforce_eager=VLLM_ENFORCE_EAGER,
        generation_config='vllm',
    )
    return llm, tokenizer


def cleanup_vllm(llm) -> None:
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def generate_responses_vllm(
    llm,
    tokenizer,
    items: Sequence[Dict[str, Any]],
    max_new_tokens: int = EVAL_MAX_NEW_TOKENS,
    temperature: float = VLLM_TEMPERATURE,
    top_p: float = VLLM_TOP_P,
    top_k: int = VLLM_TOP_K,
    repetition_penalty: float = VLLM_REPETITION_PENALTY,
) -> List[str]:
    from vllm import SamplingParams

    prompts = [build_vllm_prompt(tokenizer, item) for item in items]
    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
        max_tokens=max_new_tokens,
    )
    outputs = llm.generate(prompts, sampling_params)
    return [out.outputs[0].text.strip() if out.outputs else '' for out in outputs]


def print_eval_batch_report(stage_name: str, rows: Sequence[Dict[str, Any]], batch_rows: Sequence[Dict[str, Any]], max_examples: int = 2) -> None:
    n = len(rows)
    correct = sum(int(row.get('correct', False)) for row in rows)
    compliance = sum(int(row.get('format_ok', False)) for row in rows)
    avg_words = sum(row.get('word_count', 0) for row in rows) / n if n else 0.0
    print(f'\n[{stage_name}] after {n} eval rows: accuracy={correct / n if n else 0.0:.3f}, boxed={compliance / n if n else 0.0:.3f}, avg_words={avg_words:.1f}')
    for row in list(batch_rows)[:max_examples]:
        response = (row.get('response') or '').replace('\n', ' ')
        if len(response) > 500:
            response = response[:500] + ' ...'
        print(f"  id={row.get('id')} correct={row.get('correct')} format_ok={row.get('format_ok')} words={row.get('word_count')}")
        print(f'  response: {response}')


def evaluate_model(model_name_or_path: str, stage_name: str, eval_rows: Optional[List[Dict[str, Any]]] = None, limit: Optional[int] = None, batch_size: int = VLLM_BATCH_SIZE, report_every_batches: int = 1, report_examples: int = 2) -> Dict[str, Any]:
    if eval_rows is None:
        eval_path = EVAL_DIR / 'heldout_eval_set.jsonl'
        eval_rows = load_jsonl(eval_path) if eval_path.exists() else build_eval_sets()
    if limit:
        eval_rows = eval_rows[:limit]

    llm, tokenizer = load_vllm_engine(model_name_or_path)

    rows = []
    correct = mcq_total = mcq_correct = multi_total = multi_correct = compliance = 0
    total_words = 0

    try:
        for start in tqdm(range(0, len(eval_rows), batch_size), desc=f'Eval {stage_name}'):
            batch = eval_rows[start:start + batch_size]
            responses = generate_responses_vllm(
                llm,
                tokenizer,
                batch,
                max_new_tokens=EVAL_MAX_NEW_TOKENS,
                temperature=EVAL_VLLM_TEMPERATURE,
            )
            batch_rows = []

            for item, response in zip(batch, responses):
                is_mcq = bool(item.get('options'))
                is_multi = (not is_mcq) and count_ans_blanks(item.get('question', '')) > 1
                if item.get('eval_source') == 'public':
                    ok = score_public_item(item, response)
                else:
                    ok = score_math_item(item, response)
                fmt_ok = boxed_format_ok(item.get('question', ''), response, is_mcq=is_mcq, options=item.get('options'))
                words = response_word_count(response)

                correct += int(ok)
                compliance += int(fmt_ok)
                total_words += words
                if is_mcq:
                    mcq_total += 1
                    mcq_correct += int(ok)
                if is_multi:
                    multi_total += 1
                    multi_correct += int(ok)

                row = {
                    'id': item.get('id'),
                    'eval_source': item.get('eval_source'),
                    'is_mcq': is_mcq,
                    'is_multi': is_multi,
                    'question': item.get('question'),
                    'options': item.get('options'),
                    'gold': item.get('answer'),
                    'response': response,
                    'correct': ok,
                    'format_ok': fmt_ok,
                    'word_count': words,
                }
                rows.append(row)
                batch_rows.append(row)

            batch_index = start // batch_size + 1
            partial_metrics = {
                'n': len(rows),
                'accuracy': correct / len(rows) if rows else 0.0,
                'correct': correct,
                'mcq_accuracy': mcq_correct / mcq_total if mcq_total else None,
                'mcq_total': mcq_total,
                'multi_answer_accuracy': multi_correct / multi_total if multi_total else None,
                'multi_answer_total': multi_total,
                'avg_response_words': total_words / len(rows) if rows else 0.0,
                'boxed_compliance_rate': compliance / len(rows) if rows else 0.0,
                'inference_backend': 'vllm',
                'partial': True,
            }
            partial_out_path = EVAL_DIR / f'{stage_name}_eval_results.partial.jsonl'
            partial_summary_path = EVAL_DIR / f'{stage_name}_eval_summary.partial.json'
            write_jsonl(rows, partial_out_path)
            partial_summary_path.write_text(json.dumps(partial_metrics, indent=2, ensure_ascii=False))

            if report_every_batches and batch_index % report_every_batches == 0:
                print_eval_batch_report(stage_name, rows, batch_rows, max_examples=report_examples)
                print(f'  partial results: {partial_out_path}')
    finally:
        cleanup_vllm(llm)

    n = len(rows)
    metrics = {
        'n': n,
        'accuracy': correct / n if n else 0.0,
        'correct': correct,
        'mcq_accuracy': mcq_correct / mcq_total if mcq_total else None,
        'mcq_total': mcq_total,
        'multi_answer_accuracy': multi_correct / multi_total if multi_total else None,
        'multi_answer_total': multi_total,
        'avg_response_words': total_words / n if n else 0.0,
        'boxed_compliance_rate': compliance / n if n else 0.0,
        'inference_backend': 'vllm',
    }
    out_path = EVAL_DIR / f'{stage_name}_eval_results.jsonl'
    summary_path = EVAL_DIR / f'{stage_name}_eval_summary.json'
    write_jsonl(rows, out_path)
    summary_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
    append_results_log(stage_name, metrics)

    print(json.dumps(metrics, indent=2))
    return metrics


In [10]:
# Build or load the fixed held-out eval set.
EVAL_SET_PATH = EVAL_DIR / 'heldout_eval_set.jsonl'
if EVAL_SET_PATH.exists():
    eval_set = load_jsonl(EVAL_SET_PATH)
    print(f'Loaded existing eval set: {len(eval_set)} rows')
else: 
    eval_set = build_eval_sets(seed=SEED)


Loaded existing eval set: 200 rows


In [11]:
# Optional: set HF_TOKEN in your environment before loading gated/private Hugging Face assets.
if not os.environ.get("HF_TOKEN"):
    print("HF_TOKEN is not set; public datasets/models may still load normally.")


HF_TOKEN is not set; public datasets/models may still load normally.


## Stage 0: Format Anchors

Stage 0 training cells are intentionally removed. This notebook uses the curated Stage 0B dataset as format anchors and resumes later training from an existing merged Stage 0 checkpoint.


In [12]:
# STAGE0B_READY_TO_SFT: load the curated final format-anchor dataset.
import collections

STAGE0B_READY_TO_SFT_PATH = DATASET_DIR / 'stage0b_final_finetune.jsonl'
STAGE0B_READY_TO_SFT_MANIFEST = DATASET_DIR / 'stage0b_final_finetune_manifest.jsonl'
if not STAGE0B_READY_TO_SFT_PATH.exists():
    raise FileNotFoundError(f'Missing Stage 0B final finetune dataset: {STAGE0B_READY_TO_SFT_PATH}')

stage0_records = load_jsonl(STAGE0B_READY_TO_SFT_PATH)

if STAGE0B_READY_TO_SFT_MANIFEST.exists():
    stage0_manifest = json.loads(STAGE0B_READY_TO_SFT_MANIFEST.read_text())
    expected_rows = stage0_manifest.get('row_count')
    if expected_rows is not None and expected_rows != len(stage0_records):
        raise ValueError(f'Manifest row_count={expected_rows}, but loaded {len(stage0_records)} records')

required_keys = {'question', 'options', 'answer', 'reasoning', 'source', 'n_ans', 'is_mcq'}
missing = [(i, sorted(required_keys - set(row))) for i, row in enumerate(stage0_records, 1) if required_keys - set(row)]
if missing:
    raise ValueError(f'Stage 0B records missing required keys: {missing[:5]}')

duplicate_questions = [q for q, n in collections.Counter(row['question'] for row in stage0_records).items() if n > 1]
if duplicate_questions:
    raise ValueError(f'Stage 0B has duplicate questions; first duplicate: {duplicate_questions[0][:200]}')

bucket_counts = collections.Counter(row.get('bucket', 'unknown') for row in stage0_records)
print(f'STAGE0B_READY_TO_SFT loaded: {len(stage0_records)} records from {STAGE0B_READY_TO_SFT_PATH}')
print('Bucket counts:', dict(bucket_counts))


STAGE0B_READY_TO_SFT loaded: 872 records from /home/lucasyan/qwen3-4b-finetuning/artifacts/post_training_curriculum/datasets/stage0b_final_finetune.jsonl
Bucket counts: {'mcq10': 315, 'table_many': 98, 'single': 158, 'multi': 291, 'multi_select': 10}


## Stage 1-2: Easy-Medium Math + MCQ Mixed

Sources:
- MATH levels 1-3.
- NuminaMath easy sources: `cn_k12`, `synthetic_math`, `orca_math`, `gsm8k`, `synthetic_amc`.
- `allenai/math_qa` easy-ish MCQ subset.
- OpenMathReasoning easy/short traces when available.


In [13]:
# Stage 1-2: OpenR1 default + MathInstruct aqua_rat + Stage 0 anchors.
# Keep the canonical names (`build_stage1_2_records`, `stage1_2_records`) so downstream cells stay unchanged.
import json
from scripts.build_stage1_2_r2 import (
    ANCHOR_SOURCE as STAGE1_2_ANCHOR_SOURCE,
    DEFAULT_MAX_RENDERED_TOKENS as STAGE1_2_MAX_RENDERED_TOKENS,
    PREFERRED_MAX_ROWS as STAGE1_2_PREFERRED_MAX_ROWS,
    PREFERRED_MIN_ROWS as STAGE1_2_PREFERRED_MIN_ROWS,
    R2_NOISE_PHRASES as STAGE1_2_CLEAN_NOISE_PHRASES,
    R2_SOURCE_CAPS as STAGE1_2_SOURCE_QUOTAS,
    build_stage1_2_r2_records,
)
from scripts.export_stage1_2_review_json import build_review_rows

STAGE1_2_PATH = DATASET_DIR / 'stage1_2_r2_records.jsonl'
STAGE1_2_CLEAN_PATH = STAGE1_2_PATH
STAGE1_2_CLEAN_MANIFEST_PATH = DATASET_DIR / 'stage1_2_r2_manifest.json'
STAGE1_2_REVIEW_PATH = DATASET_DIR / 'stage1_2_r2_records_for_review.json'
STAGE1_2_ANCHOR_PATH = DATASET_DIR / 'stage0b_final_finetune.jsonl'
STAGE1_2_CLEAN_SOURCE_CAPS = STAGE1_2_SOURCE_QUOTAS
STAGE1_2_BASE_MODEL = BASE_MODEL
STAGE1_2_PREFLIGHT_MAX_TOKENS = 8192


def stage1_2_bucket_counts(records: Sequence[Dict[str, Any]]) -> Dict[str, int]:
    counts = {name: 0 for name in STAGE1_2_SOURCE_QUOTAS}
    counts[STAGE1_2_ANCHOR_SOURCE] = 0
    for row in records:
        source = str(row.get('source', ''))
        if source in counts:
            counts[source] += 1
    return counts


def stage1_2_cache_ok(records: Sequence[Dict[str, Any]], target_size: int | None = None) -> bool:
    if not records:
        return False
    counts = stage1_2_bucket_counts(records)
    allowed_sources = set(STAGE1_2_SOURCE_QUOTAS) | {STAGE1_2_ANCHOR_SOURCE}
    if any(str(row.get('source', '')) not in allowed_sources for row in records):
        return False
    if not (STAGE1_2_PREFERRED_MIN_ROWS <= len(records) <= STAGE1_2_PREFERRED_MAX_ROWS):
        return False
    if counts.get(STAGE1_2_ANCHOR_SOURCE, 0) != 1000:
        return False
    if target_size is not None and len(records) != target_size:
        return False
    return True


def build_stage1_2_records(target_size: int | None = None, seed: int = SEED, force_rebuild: bool = False) -> List[Dict[str, Any]]:
    records = build_stage1_2_r2_records(
        anchor_path=STAGE1_2_ANCHOR_PATH,
        out_path=STAGE1_2_PATH,
        manifest_path=STAGE1_2_CLEAN_MANIFEST_PATH,
        seed=seed,
        force_rebuild=force_rebuild,
        tokenizer_name_or_path=STAGE1_2_BASE_MODEL,
        max_rendered_tokens=STAGE1_2_MAX_RENDERED_TOKENS,
    )
    if target_size is not None and len(records) != target_size:
        print(f'Warning: requested target_size={target_size}, but r2 recipe produced {len(records)} records.')
    if not stage1_2_cache_ok(records):
        print(f'Stage 1-2 r2 records failed source/size sanity check: {stage1_2_bucket_counts(records)}')
    return records


def _stage1_2_filter_invalid_mcq(records: Sequence[Dict[str, Any]]) -> tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    kept = []
    dropped = []
    for idx, row in enumerate(records):
        options = row.get('options')
        is_mcq = bool(row.get('is_mcq') or options)
        if not is_mcq:
            kept.append(row)
            continue

        valid = set(option_labels(len(options or [])))
        answer = normalize_final_answer(row.get('answer', '')).strip().upper()
        assistant_text = clean_stage0_assistant_content(
            row['target'] if row.get('target') else make_assistant_content(row.get('reasoning', ''), row.get('target_answer', row.get('answer', '')))
        )
        boxes_after = extract_all_boxed(extract_after_think(assistant_text))
        box = boxes_after[0].strip().upper() if boxes_after else ''

        reasons = []
        if not options:
            reasons.append('mcq_without_options')
        if answer not in valid:
            reasons.append('mcq_answer_not_option_label')
        if box not in valid:
            reasons.append('mcq_box_not_option_label')

        if reasons:
            dropped.append({'row_index': idx, 'source': row.get('source'), 'answer': answer, 'box': box, 'valid': sorted(valid), 'reasons': reasons})
            continue
        kept.append(row)
    return kept, dropped


stage1_2_records = build_stage1_2_records(force_rebuild=False)
stage1_2_records, STAGE1_2_DROPPED_INVALID_MCQ = _stage1_2_filter_invalid_mcq(stage1_2_records)
if STAGE1_2_DROPPED_INVALID_MCQ:
    print(f'Dropped {len(STAGE1_2_DROPPED_INVALID_MCQ)} invalid Stage 1-2 MCQ rows before training.')
    print('First dropped invalid MCQ rows:', STAGE1_2_DROPPED_INVALID_MCQ[:5])
if not STAGE1_2_REVIEW_PATH.exists():
    with STAGE1_2_REVIEW_PATH.open('w') as f:
        json.dump(build_review_rows(stage1_2_records), f, ensure_ascii=False, indent=2)

STAGE1_2_TRIAL = StageConfig(name='stage1_2_trial', lr=7e-6, epochs=None, max_steps=100, lora_r=64, lora_alpha=128)
STAGE1_2 = StageConfig(name='stage1_2', lr=1e-5, epochs=1.0, max_steps=-1, lora_r=64, lora_alpha=128)

print(f'Loaded Stage 1-2 r2 records: {len(stage1_2_records)} from {STAGE1_2_PATH}')
print('Stage 1-2 r2 source counts:', stage1_2_bucket_counts(stage1_2_records))
print(f'Stage 1-2 review export path: {STAGE1_2_REVIEW_PATH}')
print('Stage 1-2 starting model:', STAGE1_2_BASE_MODEL)


Dropped 39 invalid Stage 1-2 MCQ rows before training.
First dropped invalid MCQ rows: [{'row_index': 327, 'source': 'openr1_default', 'answer': '(B)', 'box': '(B)', 'valid': ['A', 'B', 'C', 'D', 'E'], 'reasons': ['mcq_answer_not_option_label', 'mcq_box_not_option_label']}, {'row_index': 663, 'source': 'openr1_default', 'answer': '(E)', 'box': '(E)', 'valid': ['A', 'B', 'C', 'D', 'E'], 'reasons': ['mcq_answer_not_option_label', 'mcq_box_not_option_label']}, {'row_index': 983, 'source': 'openr1_default', 'answer': '(D)', 'box': '(D)', 'valid': ['A', 'B', 'C', 'D', 'E'], 'reasons': ['mcq_answer_not_option_label', 'mcq_box_not_option_label']}, {'row_index': 1025, 'source': 'openr1_default', 'answer': '(D)', 'box': '(D)', 'valid': ['A', 'B', 'C', 'D', 'E'], 'reasons': ['mcq_answer_not_option_label', 'mcq_box_not_option_label']}, {'row_index': 1096, 'source': 'openr1_default', 'answer': '(A)', 'box': '(A)', 'valid': ['A', 'B', 'C', 'D', 'E'], 'reasons': ['mcq_answer_not_option_label', 'mcq_

In [14]:
# STAGE1_2_TARGET_PREVIEW: quick check of exact assistant targets fed into SFT.
STAGE1_2_TARGET_PREVIEW_N = 3
STAGE1_2_TARGET_PREVIEW_CHAR_LIMIT = 5200


def _clip_stage1_2_preview(text, limit=STAGE1_2_TARGET_PREVIEW_CHAR_LIMIT):
    text = str(text or '').strip()
    return text if len(text) <= limit else text[:limit] + ' ...'


def _render_stage1_2_preview_question(row):
    question = str(row.get('question', '')).strip()
    options = row.get('options') or []
    if not options:
        return question
    labels = [chr(65 + i) for i in range(len(options))]
    lower_question = question.lower()
    already_has_options = any(f'({label.lower()})' in lower_question or f'{label.lower()}.' in lower_question for label in labels[:2])
    if already_has_options:
        return question
    option_lines = [f'{label}. {option}' for label, option in zip(labels, options)]
    return question + '\n' + '\n'.join(option_lines)


preview_rows = safe_shuffle_take(stage1_2_records, STAGE1_2_TARGET_PREVIEW_N, SEED + 404)
for idx, row in enumerate(preview_rows, 1):
    assistant_target = row['target'] if row.get('target') else make_assistant_content(
        row.get('reasoning', ''),
        row.get('target_answer', row.get('answer', '')),
    )
    print('=' * 80)
    print(f"TARGET PREVIEW {idx} | source={row.get('source')} | original={row.get('original_source')} | is_mcq={row.get('is_mcq')} | n_ans={row.get('n_ans')}")
    print(f"answer={row.get('answer')!r} | target_answer={row.get('target_answer', row.get('answer'))!r}")
    print('--- USER PROMPT ---')
    print(_clip_stage1_2_preview(_render_stage1_2_preview_question(row), STAGE1_2_TARGET_PREVIEW_CHAR_LIMIT))
    print('--- ASSISTANT TARGET ---')
    print(_clip_stage1_2_preview(assistant_target, STAGE1_2_TARGET_PREVIEW_CHAR_LIMIT))


TARGET PREVIEW 1 | source=openr1_default | original=openr1:olympiads | is_mcq=False | n_ans=0
answer='86' | target_answer='86'
--- USER PROMPT ---
6. How many integers $m$ in the range from 1 to 1996 are there such that $\frac{m^{2}+7}{m+4}$ is not a reduced fraction?
--- ASSISTANT TARGET ---
<think>
Okay, so I need to figure out how many integers m between 1 and 1996 make the fraction (m² + 7)/(m + 4) not reduced. Hmm, a fraction is not reduced when the numerator and denominator share a common factor greater than 1. So, I need to find all m where the greatest common divisor (gcd) of m² + 7 and m + 4 is greater than 1. 

Let me start by recalling that gcd(a, b) = gcd(b, a mod b). Maybe I can use the Euclidean algorithm here. Let me set a = m² + 7 and b = m + 4. Then, compute a mod b. 

First, divide m² + 7 by m + 4. Let me do polynomial long division. 

m² divided by m is m, so multiply (m + 4) by m to get m² + 4m. Subtract that from m² + 7: (m² + 7) - (m² + 4m) = -4m + 7. 

Now, divid

In [15]:
# STAGE1_2_PRETRAIN_INSPECT: verify cleaned Stage 1-2 data/config before training.
import collections
import json
import statistics
from dataclasses import asdict
from pathlib import Path

STAGE1_2_TOKEN_SCAN_LIMIT = None  # None = scan every record; set 1000 for a quicker spot check.
STAGE1_2_PREVIEW_N = 3
STAGE1_2_MAX_ERROR_EXAMPLES = 8


def _short_preview(text, limit=1400):
    text = str(text or '')
    return text if len(text) <= limit else text[:limit] + ' ...'


def _assistant_text_for_preflight(row):
    if row.get('target'):
        return clean_stage0_assistant_content(row['target'])
    return clean_stage0_assistant_content(make_assistant_content(row.get('reasoning', ''), row.get('target_answer', row.get('answer', ''))))


def _word_count(text):
    return len(re.findall(r'\S+', str(text or '')))


def _stage1_2_word_stats(values):
    values = sorted(values)
    if not values:
        return None
    p95 = values[int(0.95 * (len(values) - 1))]
    return {'min': values[0], 'median': int(statistics.median(values)), 'p95': p95, 'max': values[-1]}


def _stage1_2_preflight(records, tokenizer, token_scan_limit=None, max_tokens=8192):
    if not records:
        raise RuntimeError('stage1_2_records is empty. Run the Stage 1-2 clean build cell first.')

    required = {'question', 'answer', 'source', 'n_ans', 'is_mcq'}
    source_counts = collections.Counter(str(r.get('source', 'unknown')) for r in records)
    mcq_count = sum(1 for r in records if r.get('is_mcq') or r.get('options'))
    n_ans_counts = collections.Counter(int(r.get('n_ans') or 0) for r in records)
    format_errors = []
    format_error_counts = collections.Counter()
    requested_in_targets = 0
    internal_boxes = think_tagged_solutions = remaining_noise = 0
    non_anchor_word_lengths = collections.defaultdict(list)
    noise_phrases = globals().get('STAGE1_2_CLEAN_NOISE_PHRASES', [])
    anchor_source = globals().get('STAGE1_2_ANCHOR_SOURCE', 'stage0_format_anchor')

    manifest_path = globals().get('STAGE1_2_CLEAN_MANIFEST_PATH')
    if manifest_path and Path(manifest_path).exists():
        manifest = json.loads(Path(manifest_path).read_text())
        print('Clean dataset manifest:', Path(manifest_path))
        print('Manifest source counts:', manifest.get('source_counts'))
        print('Manifest accepted by source:', manifest.get('accepted_by_source'))
        print('Manifest top drops:', dict(collections.Counter(manifest.get('drop_counts', {})).most_common(20)))
        print('Manifest word stats:', manifest.get('word_counts_by_source'))

    for idx, row in enumerate(records):
        missing = sorted(k for k in required if k not in row)
        if row.get('source') != anchor_source and 'reasoning' not in row:
            missing.append('reasoning')
        if missing:
            format_error_counts['missing_keys'] += 1
            format_errors.append((idx, 'missing_keys', missing, row.get('source')))
            continue

        question = str(row.get('question') or '').strip()
        answer = normalize_final_answer(row.get('answer', '')).strip()
        options = row.get('options')
        is_mcq = bool(row.get('is_mcq') or options)
        assistant_text = _assistant_text_for_preflight(row)
        after_think = extract_after_think(assistant_text)
        boxes_after = extract_all_boxed(after_think)

        if row.get('source') != anchor_source:
            solution = str(row.get('reasoning', ''))
            low_solution = solution.lower()
            non_anchor_word_lengths[str(row.get('source'))].append(_word_count(solution))
            internal_boxes += int('\\boxed' in solution)
            think_tagged_solutions += int('<think>' in solution or '</think>' in solution)
            remaining_noise += int(any(phrase.lower() in low_solution for phrase in noise_phrases))

        def add_error(kind, detail=''):
            format_error_counts[kind] += 1
            if len(format_errors) < STAGE1_2_MAX_ERROR_EXAMPLES:
                format_errors.append((idx, kind, detail, row.get('source')))

        if not question:
            add_error('empty_question')
        if not answer:
            add_error('empty_answer')
        if int(row.get('n_ans') or 0) != count_ans_blanks(question):
            add_error('n_ans_mismatch', {'stored': row.get('n_ans'), 'actual': count_ans_blanks(question)})
        if assistant_text.count('<think>') != 1:
            add_error('bad_think_open_count', assistant_text.count('<think>'))
        if assistant_text.count('</think>') != 1:
            add_error('bad_think_close_count', assistant_text.count('</think>'))
        if len(boxes_after) != 1:
            add_error('boxed_after_think_count', len(boxes_after))
        elif not boxes_after[0].strip():
            add_error('empty_box')

        if is_mcq:
            valid = set(option_labels(len(options or [])))
            if not options:
                add_error('mcq_without_options')
            if answer.upper() not in valid:
                add_error('mcq_answer_not_option_label', {'answer': answer, 'valid': sorted(valid)})
            if boxes_after and boxes_after[0].strip().upper() not in valid:
                add_error('mcq_box_not_option_label', {'box': boxes_after[0], 'valid': sorted(valid)})
        else:
            expected = count_ans_blanks(question)
            if expected > 1 and boxes_after and len(split_final_answers(boxes_after[0])) != expected:
                add_error('multi_answer_count_mismatch', {'expected': expected, 'boxed': boxes_after[0]})

        if 'requested' in assistant_text.lower():
            requested_in_targets += assistant_text.lower().count('requested')

    print('Stage 1-2 starting model:', STAGE1_2_BASE_MODEL)
    print('Stage 1-2 trial config:', asdict(STAGE1_2_TRIAL))
    print('Stage 1-2 full config:', asdict(STAGE1_2))
    print(f'Tokenizer EOS/PAD: {tokenizer.eos_token!r}/{tokenizer.pad_token!r} ids={tokenizer.eos_token_id}/{tokenizer.pad_token_id}')
    print(f'Record count: {len(records)}')
    print(f'MCQ rows: {mcq_count}')
    print('n_ans counts:', dict(n_ans_counts))
    print('Source counts:', dict(source_counts))
    print('Non-anchor word stats:', {src: _stage1_2_word_stats(vals) for src, vals in non_anchor_word_lengths.items()})
    print(f'Internal boxes in cleaned non-anchor solutions: {internal_boxes}')
    print(f'Think tags in cleaned non-anchor solutions: {think_tagged_solutions}')
    print(f'Remaining noise phrases in cleaned non-anchor solutions: {remaining_noise}')
    warning_format_error_kinds = {'multi_answer_count_mismatch'}
    hard_format_error_counts = collections.Counter({
        kind: count for kind, count in format_error_counts.items()
        if kind not in warning_format_error_kinds
    })
    print(f'Raw/rendered target format errors: {sum(format_error_counts.values())}')
    print('Format error counts:', dict(format_error_counts))
    print('Hard format error counts:', dict(hard_format_error_counts))
    print('Warning-only format errors:', {kind: format_error_counts[kind] for kind in warning_format_error_kinds if format_error_counts.get(kind)})
    print(f'Requested occurrences in rendered targets after cleaning: {requested_in_targets}')

    scan_records = records if token_scan_limit is None else records[:min(token_scan_limit, len(records))]
    lengths = []
    trainable_lengths = []
    overlength = []
    token_errors = []
    eos_labeled = 0
    requested_in_labeled_targets = 0

    for pos, row in enumerate(scan_records, 1):
        if pos % 5000 == 0:
            print(f'Token scan progress: {pos}/{len(scan_records)}')
        try:
            item = make_assistant_only_features(tokenizer, row)
            labels = item['labels']
            first_train_idx = next((i for i, label in enumerate(labels) if label != -100), None)
            trainable_ids = [tok for tok, label in zip(item['input_ids'], labels) if label != -100]
            target_text = tokenizer.decode(trainable_ids, skip_special_tokens=False)
            lengths.append(len(item['input_ids']))
            trainable_lengths.append(len(trainable_ids))
            if len(item['input_ids']) > max_tokens:
                overlength.append((pos - 1, len(item['input_ids']), row.get('source')))
            if first_train_idx is None or not trainable_ids:
                token_errors.append((pos - 1, 'no_trainable_tokens', row.get('source')))
            if '</think>' not in target_text or '\\boxed' not in target_text:
                token_errors.append((pos - 1, 'missing_think_close_or_box_in_labeled_target', row.get('source')))
            if trainable_ids and tokenizer.eos_token_id is not None and trainable_ids[-1] == tokenizer.eos_token_id:
                eos_labeled += 1
            requested_in_labeled_targets += target_text.lower().count('requested')
        except Exception as exc:
            token_errors.append((pos - 1, type(exc).__name__, str(exc)[:240]))

    print(f'Token-scanned rows: {len(scan_records)}')
    if lengths:
        p95 = int(statistics.quantiles(lengths, n=20)[18]) if len(lengths) >= 20 else max(lengths)
        print(f'Token lengths: min={min(lengths)} median={int(statistics.median(lengths))} p95={p95} max={max(lengths)}')
        print(f'Labeled assistant tokens: min={min(trainable_lengths)} median={int(statistics.median(trainable_lengths))} max={max(trainable_lengths)}')
    print(f'Overlength rows > preflight max tokens={max_tokens}: {len(overlength)}')
    print(f'Labeled target ends with EOS: {eos_labeled}/{len(scan_records)}')
    print(f'Requested occurrences in labeled targets: {requested_in_labeled_targets}')
    print(f'Tokenization/label errors: {len(token_errors)}')

    if format_errors:
        print('\nFirst rendered-format error examples:')
        for err in format_errors[:STAGE1_2_MAX_ERROR_EXAMPLES]:
            print(' ', err)
    if overlength[:STAGE1_2_MAX_ERROR_EXAMPLES]:
        print('\nFirst overlength rows:', overlength[:STAGE1_2_MAX_ERROR_EXAMPLES])
    if token_errors[:STAGE1_2_MAX_ERROR_EXAMPLES]:
        print('\nFirst tokenization/label errors:', token_errors[:STAGE1_2_MAX_ERROR_EXAMPLES])

    for preview_idx, row in enumerate(records[:STAGE1_2_PREVIEW_N], 1):
        item = make_assistant_only_features(tokenizer, row)
        first_train_idx = next(i for i, label in enumerate(item['labels']) if label != -100)
        prompt_text = tokenizer.decode(item['input_ids'][:first_train_idx], skip_special_tokens=False)
        target_text = tokenizer.decode(item['input_ids'][first_train_idx:], skip_special_tokens=False)
        trainable_tokens = sum(1 for label in item['labels'] if label != -100)
        print('\n' + '=' * 80)
        print(f'PREVIEW {preview_idx} | source={row.get("source")} | is_mcq={row.get("is_mcq")} | n_ans={row.get("n_ans")}')
        print(f'tokens: total={len(item["input_ids"])} prompt_masked={first_train_idx} assistant_trainable={trainable_tokens}')
        print('--- PROMPT CONTEXT (masked; no loss) ---')
        print(_short_preview(prompt_text[-1200:]))
        print('--- ASSISTANT TARGET (loss applies) ---')
        print(_short_preview(target_text, 1800))

    hard_fail = False
    if hard_format_error_counts or token_errors:
        hard_fail = True
    if len(overlength) / max(1, len(scan_records)) > 0.05:
        hard_fail = True
    if requested_in_labeled_targets or internal_boxes or think_tagged_solutions or remaining_noise:
        hard_fail = True

    if hard_fail:
        raise RuntimeError('STAGE1_2_PRETRAIN_INSPECT failed. Fix the reported data/label issues before training.')

    print('\nSTAGE1_2_PRETRAIN_INSPECT PASS: run the Stage 1-2 full training cell.')


stage1_2_preview_tokenizer = load_tokenizer_only(STAGE1_2_BASE_MODEL)
_stage1_2_preflight(stage1_2_records, stage1_2_preview_tokenizer, STAGE1_2_TOKEN_SCAN_LIMIT, STAGE1_2_PREFLIGHT_MAX_TOKENS)



tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Clean dataset manifest: /home/lucasyan/qwen3-4b-finetuning/artifacts/post_training_curriculum/datasets/stage1_2_r2_manifest.json
Manifest source counts: {'openr1_default': 6498, 'stage0_format_anchor': 1000, 'mathinstruct_aqua_rat': 1000}
Manifest accepted by source: {'openr1_default': 6500, 'mathinstruct_aqua_rat': 1000}
Manifest top drops: {'openr1:no_clean_verified_generation': 54725, 'mathinstruct:too_short': 3222, 'mathinstruct:answer_extract_fail': 1989, 'mathinstruct:abrupt_tail': 21, 'mathinstruct:noise_phrase': 8, 'mathinstruct:codey_output': 7, 'mathinstruct:empty_reasoning': 2, 'validation:invalid_answer_field': 2, 'mathinstruct:missing_question_or_options': 1}
Manifest word stats: {'openr1_default': {'min': 50, 'median': 535, 'max': 700}, 'mathinstruct_aqua_rat': {'min': 50, 'median': 67, 'max': 460}}
Stage 1-2 starting model: Qwen/Qwen3-4B-Thinking-2507
Stage 1-2 trial config: {'name': 'stage1_2_trial', 'lr': 7e-06, 'epochs': None, 'max_steps': 100, 'lora_r': 64, 'lora_alp

### Stage 1-2 Sample Log

Run the next cell to print the exact Stage 1-2 distribution and representative rendered samples before launching the trial gate.


In [16]:
# STAGE1_2_SAMPLE_LOG: clean source distribution and representative rendered targets.
import collections
import json

STAGE1_2_SAMPLE_N = 1
STAGE1_2_SAMPLE_CHAR_LIMIT = 1600


def _clip_sample_text(text, limit=STAGE1_2_SAMPLE_CHAR_LIMIT):
    text = str(text or '').strip()
    return text if len(text) <= limit else text[:limit] + ' ...'


def _render_stage1_2_sample(row):
    user_prompt = build_user_problem(row['question'], row.get('options'))
    assistant_target = _assistant_text_for_preflight(row) if '_assistant_text_for_preflight' in globals() else (
        row['target'] if row.get('target') else make_assistant_content(row.get('reasoning', ''), row.get('target_answer', row.get('answer', '')))
    )
    return user_prompt, assistant_target


def log_stage1_2_samples(records, sample_n=STAGE1_2_SAMPLE_N):
    source_counts = collections.Counter(str(r.get('source', 'unknown')) for r in records)
    mcq_count = sum(bool(r.get('is_mcq') or r.get('options')) for r in records)
    n_ans_counts = collections.Counter(int(r.get('n_ans') or 0) for r in records)

    print(f'Record count: {len(records)}')
    print('Clean source caps:', globals().get('STAGE1_2_CLEAN_SOURCE_CAPS'))
    print('Actual clean sources:', dict(source_counts))
    print(f'MCQ rows: {mcq_count}')
    print('n_ans counts:', dict(n_ans_counts))
    if 'STAGE1_2_CLEAN_MANIFEST_PATH' in globals() and Path(STAGE1_2_CLEAN_MANIFEST_PATH).exists():
        manifest = json.loads(Path(STAGE1_2_CLEAN_MANIFEST_PATH).read_text())
        print('Manifest top drops:', dict(collections.Counter(manifest.get('drop_counts', {})).most_common(20)))
        print('Manifest word stats:', manifest.get('word_counts_by_source'))

    if not (8000 <= len(records) <= 9300):
        raise RuntimeError(f'Clean Stage 1-2 row count outside preferred range: {len(records)}')
    if source_counts.get('math_qa_easy', 0):
        raise RuntimeError('Clean Stage 1-2 data should not contain math_qa_easy rows.')

    for source_idx, source in enumerate(sorted(source_counts)):
        source_rows = [r for r in records if str(r.get('source', 'unknown')) == source]
        samples = safe_shuffle_take(source_rows, sample_n, SEED + 2000 + source_idx)
        print('\n' + '=' * 100)
        print(f'{source}: showing {len(samples)} / {len(source_rows)} records')
        for sample_idx, row in enumerate(samples, 1):
            user_prompt, assistant_target = _render_stage1_2_sample(row)
            print('\n' + '-' * 80)
            print(f"SAMPLE {sample_idx} | source={row.get('source')} | original={row.get('original_source')} | is_mcq={row.get('is_mcq')} | n_ans={row.get('n_ans')} | answer={row.get('answer')!r} | target_answer={row.get('target_answer', row.get('answer'))!r}")
            print('--- USER PROMPT ---')
            print(_clip_sample_text(user_prompt, 900))
            print('--- ASSISTANT TARGET ---')
            print(_clip_sample_text(assistant_target, 2400))


log_stage1_2_samples(stage1_2_records)



Record count: 8459
Clean source caps: {'openr1_default': 6500, 'mathinstruct_aqua_rat': 1500}
Actual clean sources: {'openr1_default': 6459, 'stage0_format_anchor': 1000, 'mathinstruct_aqua_rat': 1000}
MCQ rows: 2177
n_ans counts: {0: 7829, 2: 185, 1: 196, 3: 102, 5: 27, 4: 50, 15: 3, 6: 26, 12: 3, 10: 3, 11: 1, 8: 11, 9: 5, 7: 14, 13: 1, 24: 1, 14: 1, 42: 1}
Manifest top drops: {'openr1:no_clean_verified_generation': 54725, 'mathinstruct:too_short': 3222, 'mathinstruct:answer_extract_fail': 1989, 'mathinstruct:abrupt_tail': 21, 'mathinstruct:noise_phrase': 8, 'mathinstruct:codey_output': 7, 'mathinstruct:empty_reasoning': 2, 'validation:invalid_answer_field': 2, 'mathinstruct:missing_question_or_options': 1}
Manifest word stats: {'openr1_default': {'min': 50, 'median': 535, 'max': 700}, 'mathinstruct_aqua_rat': {'min': 50, 'median': 67, 'max': 460}}

mathinstruct_aqua_rat: showing 1 / 1000 records

--------------------------------------------------------------------------------
SAMPLE

In [17]:
print(len(stage1_2_records))

8459


In [18]:
# # Stage 1-2 trial gate: 400 steps, then evaluate on the fixed 50 public rows if available.
# STAGE1_2_TRIAL.lr = 7e-6
# STAGE1_2_TRIAL.max_steps = 150
# STAGE1_2_TRIAL.lora_r = 64
# STAGE1_2_TRIAL.lora_alpha = 128
# lora_trial, merged_trial = train_stage(STAGE1_2_BASE_MODEL, stage1_2_records, STAGE1_2_TRIAL, merge=True)



In [19]:
# if 'merged_trial' not in globals() or merged_trial is None or not Path(merged_trial).exists():
#     merged_trial = CHECKPOINT_DIR / 'merged_stage1_2_trial'
# if not Path(merged_trial).exists():
#     raise FileNotFoundError(f'Merged Stage 1-2 trial checkpoint not found at {merged_trial}. Run the trial training cell first.')

# public_eval_50 = [row for row in eval_set if row.get('eval_source') == 'public'][:50]
# trial_metrics = evaluate_model(str(merged_trial), 'stage1_2_trial_public50', public_eval_50 or eval_set[:50])

# if trial_metrics['mcq_total'] and trial_metrics['mcq_accuracy'] is not None and trial_metrics['mcq_accuracy'] < 0.50:
#     raise RuntimeError('Stage 1-2 trial MCQ accuracy below 50%; stop and inspect data/prompt mix.')
# if trial_metrics['accuracy'] <= 0.56:
#     raise RuntimeError('Stage 1-2 trial did not beat the 56% vanilla baseline gate.')

In [20]:
# preview_tokenizer = load_tokenizer_only(STAGE1_2_BASE_MODEL)
# stage1_2_formatted = format_records_with_tokenizer(stage1_2_records, preview_tokenizer)

# for i in range(3):
#     sample = stage1_2_formatted[i]
#     first_train_idx = next(j for j, label in enumerate(sample["labels"]) if label != -100)
#     target_text = preview_tokenizer.decode(sample["input_ids"][first_train_idx:], skip_special_tokens=False)

#     print("=== SAMPLE", i, "===")
#     print("SOURCE:", sample.get("source"))
#     print("ANSWER:", sample.get("answer"))
#     print("LAST 300 CHARS OF TARGET:")
#     print(target_text[-300:])
#     print()


In [21]:
# Stage 1-2 full run -> eval -> merge.
lora_stage1_2, merged_stage1_2 = train_stage(STAGE1_2_BASE_MODEL, stage1_2_records, STAGE1_2, merge=True)
stage1_2_metrics = evaluate_model(str(merged_stage1_2), 'stage1_2', eval_set)


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading model for stage1_2: Qwen/Qwen3-4B-Thinking-2507
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_10189/421846491.py:26: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.19.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

...

Tokenizer EOS/PAD: '<|im_end|>' / '<|im_end|>'
Skipped 8 over-length records; first skipped: (212, 9890, None)
Assistant-only train records: 8451 / 8459


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,451 | Num Epochs = 1 | Total steps = 529
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,0.649200
10,0.708600
15,0.621600
20,0.491100
25,0.523600
30,0.454100


KeyboardInterrupt: 

## Stage 2.5: Rejection Sampling From Stage 2 Checkpoint

Generate self-traces from `merged_stage1_2`, score them against known-answer prompts, and keep correct traces only.


In [ ]:
def run_rejection_sampling(
    model_name_or_path: str,
    seed_records: Sequence[Dict[str, Any]],
    out_path: Path,
    target_size: int,
    stage_name: str,
    candidates_per_problem: int = 2,
    batch_size: int = VLLM_BATCH_SIZE,
    temperature: float = 0.0,
    top_p: float = 1.0,
) -> List[Dict[str, Any]]:
    llm, tokenizer = load_vllm_engine(model_name_or_path)

    accepted = []
    pool = safe_shuffle_take(seed_records, len(seed_records), SEED + 17)

    try:
        for start in tqdm(range(0, len(pool), batch_size), desc=f'Rejection sampling {stage_name}'):
            if len(accepted) >= target_size:
                break

            batch = pool[start:start + batch_size]
            generation_items = []
            source_rows = []
            for row in batch:
                for _ in range(candidates_per_problem):
                    generation_items.append({'question': row['question'], 'options': row.get('options'), 'answer': row['answer']})
                    source_rows.append(row)

            responses = generate_responses_vllm(
                llm,
                tokenizer,
                generation_items,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=temperature,
                top_p=top_p,
            )

            seen_in_batch = set()
            for row, response in zip(source_rows, responses):
                if len(accepted) >= target_size:
                    break
                row_key = id(row)
                if row_key in seen_in_batch:
                    continue
                pseudo_item = {'question': row['question'], 'options': row.get('options'), 'answer': row['answer']}
                ok = score_public_item(pseudo_item, response)
                if ok and boxed_format_ok(row['question'], response, bool(row.get('options')), row.get('options')):
                    reasoning = response.split('</think>')[0].replace('<think>', '').strip() if '</think>' in response else response
                    accepted.append(make_training_record(row['question'], row['answer'], reasoning, row.get('options'), source=f'{stage_name}_accepted'))
                    seen_in_batch.add(row_key)
    finally:
        cleanup_vllm(llm)

    write_jsonl(accepted, out_path)
    print(f'Accepted {len(accepted)} / target {target_size}: {out_path}')
    return accepted

stage2_5_records = run_rejection_sampling(
    model_name_or_path=str(merged_stage1_2),
    seed_records=stage1_2_records,
    out_path=DATASET_DIR / 'stage2_5_rejection_records.jsonl',
    target_size=5000,
    stage_name='stage2_5',
)

In [ ]:
lora_stage2_5, merged_stage2_5 = train_stage(str(merged_stage1_2), stage2_5_records, STAGE2_5, merge=True)
stage2_5_metrics = evaluate_model(str(merged_stage2_5), 'stage2_5', eval_set)


## Stage 3-4: Hard / Olympiad Ceiling

Sources:
- MATH levels 4-5.
- NuminaMath olympiad, AOPS, AMC/AIME-like sources.
- OpenMathReasoning hard/long traces.
- `simplescaling/s1K-1.1` using `deepseek_thinking_trajectory` as CoT and `deepseek_attempt` as answer source.


In [ ]:
def build_stage3_4_records(target_size: int = 25000, seed: int = SEED) -> List[Dict[str, Any]]:
    records = []

    try:
        math_train = load_dataset('HuggingFaceH4/MATH', split='train')
        for ex in math_train:
            lvl = math_level_value(ex)
            if lvl is not None and lvl >= 4:
                sol = ex.get('solution', '')
                records.append(make_training_record(ex.get('problem', ''), extract_answer_from_solution(sol), sol, source=f'math_l{lvl}'))
    except Exception as exc:
        print(f'Warning: could not load hard MATH: {exc}')

    try:
        numina = load_dataset('AI-MO/NuminaMath-CoT', split='train')
        hard_keys = ['olympiad', 'aops', 'aime', 'amc']
        for ex in numina:
            src = str(ex.get('source', '')).lower()
            if any(k in src for k in hard_keys):
                problem = ex.get('problem') or ex.get('question') or ''
                sol = ex.get('solution') or ex.get('answer') or ''
                records.append(make_training_record(problem, extract_answer_from_solution(sol), sol, source=f'numina_{src}'))
    except Exception as exc:
        print(f'Warning: could not load hard NuminaMath: {exc}')

    try:
        omr = load_dataset('nvidia/OpenMathReasoning', split='train')
        for ex in omr.shuffle(seed=seed).select(range(min(8000, len(omr)))):
            problem = ex.get('problem') or ex.get('question') or ex.get('input') or ''
            sol = ex.get('solution') or ex.get('reasoning') or ex.get('output') or ''
            if response_word_count(sol) > 600:
                records.append(make_training_record(problem, extract_answer_from_solution(sol), sol, source='openmathreasoning_hard'))
    except Exception as exc:
        print(f'Warning: could not load OpenMathReasoning hard: {exc}')

    try:
        s1k = load_dataset('simplescaling/s1K-1.1', split='train')
        for ex in s1k:
            problem = ex.get('question') or ex.get('problem') or ex.get('input') or ''
            cot = ex.get('deepseek_thinking_trajectory') or ex.get('thinking_trajectory') or ''
            attempt = ex.get('deepseek_attempt') or ex.get('answer') or ex.get('solution') or ''
            final = extract_answer_from_solution(attempt) or extract_answer_from_solution(cot)
            records.append(make_training_record(problem, final, cot or attempt, source='s1k_deepseek'))
    except Exception as exc:
        print(f'Warning: could not load s1K: {exc}')

    records = safe_shuffle_take(records, target_size, seed)
    write_jsonl(records, DATASET_DIR / 'stage3_4_records.jsonl')
    print(f'Stage 3-4 records: {len(records)}')
    return records

stage3_4_records = build_stage3_4_records(target_size=25000)


In [ ]:
lora_stage3_4, merged_stage3_4 = train_stage(str(merged_stage2_5), stage3_4_records, STAGE3_4, merge=True)
stage3_4_metrics = evaluate_model(str(merged_stage3_4), 'stage3_4', eval_set)


## Stage 3.5: Hard-Tier Rejection Sampling

Generate correct self-traces from the Stage 3-4 checkpoint and train one extra hard self-distillation pass.


In [ ]:
stage3_5_records = run_rejection_sampling(
    model_name_or_path=str(merged_stage3_4),
    seed_records=stage3_4_records,
    out_path=DATASET_DIR / 'stage3_5_rejection_records.jsonl',
    target_size=3000,
    stage_name='stage3_5',
)

lora_stage3_5, merged_stage3_5 = train_stage(str(merged_stage3_4), stage3_5_records, STAGE3_5, merge=True)
stage3_5_metrics = evaluate_model(str(merged_stage3_5), 'stage3_5', eval_set)


## Stage 5: Stratos + Stratified Replay

Sources:
- 10% Bespoke-Stratos-17k.
- 10% stratified replay from all prior stages.

Goal: stabilize, reduce forgetting, preserve format/MCQ behavior.


In [ ]:
def build_stage5_records(seed: int = SEED) -> List[Dict[str, Any]]:
    rng = random.Random(seed)
    records = []

    try:
        stratos = load_dataset('bespokelabs/Bespoke-Stratos-17k', split='train')
        n = max(1, int(0.10 * len(stratos)))
        for ex in stratos.shuffle(seed=seed).select(range(n)):
            problem = ex.get('prompt') or ex.get('question') or ex.get('problem') or ex.get('input') or ''
            sol = ex.get('response') or ex.get('solution') or ex.get('output') or ''
            records.append(make_training_record(problem, extract_answer_from_solution(sol), sol, source='stratos_10pct'))
    except Exception as exc:
        print(f'Warning: could not load Bespoke-Stratos-17k: {exc}')

    prior_groups = [stage0_records, stage1_2_records, stage2_5_records, stage3_4_records, stage3_5_records]
    for i, group in enumerate(prior_groups):
        n = max(1, int(0.10 * len(group))) if group else 0
        records.extend(safe_shuffle_take(group, n, seed + 100 + i))

    records = safe_shuffle_take(records, 17000, seed)
    write_jsonl(records, DATASET_DIR / 'stage5_records.jsonl')
    print(f'Stage 5 records: {len(records)}')
    return records

stage5_records = build_stage5_records()


In [ ]:
lora_stage5, merged_stage5 = train_stage(str(merged_stage3_5), stage5_records, STAGE5, merge=True)
stage5_metrics = evaluate_model(str(merged_stage5), 'stage5_final', eval_set)
print(f'Final merged model saved at: {merged_stage5}')


## Final Smoke Test

Use this after Stage 5 to inspect actual generation format before packaging/submission.


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(merged_stage5),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

smoke_item = {
    'question': 'For the equation $y=6.5x-6$, a. the $y$-intercept is [ANS], and the slope is [ANS]. b. the line [ANS] A. slopes upward  B. is horizontal  C. slopes downward  D. none of the above',
    'options': None,
}
print(generate_response(model, tokenizer, smoke_item, max_new_tokens=1024))
